## Locally Linear Approximation

In this notebook, locally linear SDE is modeled with its drift and control tensor being learnable neural net

$$
    \forall t \in [t_i ,t_{i+1}) \;\;dX_t = (-A_iX_t + \alpha_i)dt + dW_t, \quad A \in \text{PSD}(\mathbb{R}^{d\times d})
$$

Matrix $A$ and the vector $\alpha$ is modeled with NN and matrix with parameter $\theta$

$$
    A_i = \sum_k w_{ik} A_k^{(i)}, \quad \alpha_i = B_\theta z_i
$$

Where,
$$
    w_{ik} = \text{softmax}(f_\theta(z_i))
$$

note that the matrix $A$ is PSD. Therefore, it is decomposed to diagonal and orthogonal matrix
$$
    A_i = E^\intercal \text{diag}\{\lambda^{(i)}\}E , \quad E \in O(\mathbb{R}^{d \times d})
$$


The implementation step is as follows:

- make NN for $f_\theta(\cdot)$ and imply softmax to the reutrn value

- from NN structure -> eval w -> eval A  ( All matrix shares the common eigen basis $E$ )

- eval B matrix




In [4]:

from src import make_ou_data
from src import ObsEncoder
from src import ACSSMTransformer, masking_unseen_data_Full_Assim
from src import PSDLocalLinearApprox

# 1. Data preparation
data = make_ou_data(32, 40, 3)
obs = data["observation"]
obs_mask = data["obs_mask"]
times = data["times"]

# 2. Encoding the observation
encoder = ObsEncoder(obs_dim= obs.shape[-1], embed_dim=8)
sampled_latent_obs = encoder(obs, obs_mask)
B,T,D =sampled_latent_obs.shape # (32, 40, 8)

# 2.1 Transformer layer
transformer = ACSSMTransformer(D,2)
z_raw = transformer(sampled_latent_obs, times, obs_mask)
z = masking_unseen_data_Full_Assim(z_raw, obs_mask)

# 3. Local linear approximation
localLinApp = PSDLocalLinearApprox(D,D)
linear_SDE_param = localLinApp(z)



time_embedding: torch.Size([32, 40, 8])
transformer_input: torch.Size([32, 40, 8])
z_raw: torch.Size([32, 40, 8])


RuntimeError: The size of tensor a (4) must match the size of tensor b (40) at non-singleton dimension 1